# 🛡️ AGL Security Tool — 5-Layer Pipeline Test

**اختبار الطبقات الخمس المنفصلة المتصلة:**
- L1: `state_extraction` — استخراج الحالة المالية
- L2: `action_space` — فضاء الأفعال
- L3: `attack_engine` — محاكاة الهجمات
- L4: `search_engine` — بحث MCTS/Beam/Evolutionary
- L5: `detectors` — 22 كاشف مستقل

**Colab Free: 12.7GB RAM — أكثر من كافي**

## 📦 Step 1: Upload Project

ارفع ملف `agl_security_tool.zip` المضغوط (فقط مجلدات `agl_security_tool` + `test_project`)

In [ ]:
# === Option A: Upload ZIP ===
# Run this cell, then upload agl_upload.zip when prompted
from google.colab import files
import os, zipfile

print("\u2935\ufe0f Upload agl_upload.zip (will be created in next cell if needed)")
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/AGL')
        print(f"\u2705 Extracted {fname} to /content/AGL")

os.chdir('/content/AGL')
!ls -la

In [ ]:
# === Option B: Clone from GitHub (if you have a repo) ===
# Uncomment and edit the URL:
# !git clone https://github.com/YOUR_USER/AGL.git /content/AGL
# %cd /content/AGL

## ⚙️ Step 2: Install Dependencies

In [ ]:
!pip install z3-solver psutil -q
print("\u2705 Dependencies installed")

## 📋 Step 3: Verify Project Structure

In [ ]:
import os, sys

# Set working directory
if os.path.exists('/content/AGL'):
    os.chdir('/content/AGL')
elif os.path.exists('/content/AGL/AGL'):
    os.chdir('/content/AGL/AGL')

sys.path.insert(0, os.getcwd())

# Verify structure
checks = [
    'agl_security_tool/__init__.py',
    'agl_security_tool/state_extraction/__init__.py',
    'agl_security_tool/action_space/__init__.py',
    'agl_security_tool/attack_engine/__init__.py',
    'agl_security_tool/search_engine/__init__.py',
    'agl_security_tool/detectors/__init__.py',
    'test_project/src/Vault.sol',
]

all_ok = True
for f in checks:
    exists = os.path.exists(f)
    status = '\u2705' if exists else '\u274c'
    print(f'  {status} {f}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n\u2705 All files present! Ready to test.')
else:
    print('\n\u274c Some files missing. Check your upload.')

# Show RAM
import psutil
ram = psutil.virtual_memory()
print(f'\n\ud83d\udcca RAM: {ram.total/1024**3:.1f} GB total, {ram.available/1024**3:.1f} GB free')

---
## 🟢 Layer 1: State Extraction

**استخراج الحالة المالية** — كيانات وعلاقات وتدفقات مالية

Sub-modules: `entity_extractor`, `relationship_mapper`, `fund_mapper`

In [ ]:
import time, gc, json, psutil

TARGET = 'test_project/src/Vault.sol'

def mem_mb():
    return psutil.Process().memory_info().rss / 1024**2

with open(TARGET, 'r') as f:
    source_code = f.read()

print(f'Source: {len(source_code)} chars, {len(source_code.splitlines())} lines')
print(f'RAM: {mem_mb():.0f} MB')
print('=' * 60)

### 1a. Entity Extraction

In [ ]:
t0 = time.time()
from agl_security_tool.state_extraction.entity_extractor import EntityExtractor

extractor = EntityExtractor()
entities = extractor.extract(source_code, TARGET)
t1 = time.time()

print(f'\u23f1 Time: {t1-t0:.3f}s | RAM: {mem_mb():.0f} MB')
print(f'\ud83d\udfe2 Entities found: {len(entities)}')
print()
for e in entities:
    conf = getattr(e, 'confidence', 0)
    print(f'  \u2022 {e.name:30s} | type={str(e.entity_type):15s} | conf={conf:.2f}')

# Save for later layers
_entities = entities
del extractor; gc.collect()

### 1b. Relationship Mapping

In [ ]:
t0 = time.time()
from agl_security_tool.state_extraction.relationship_mapper import RelationshipMapper

mapper = RelationshipMapper()
relationships = mapper.map(source_code, TARGET)
t1 = time.time()

print(f'\u23f1 Time: {t1-t0:.3f}s | RAM: {mem_mb():.0f} MB')
print(f'\ud83d\udfe2 Relationships found: {len(relationships)}')
print()
categories = {}
for r in relationships:
    cat = r.category
    categories[cat] = categories.get(cat, 0) + 1
    print(f'  \u2022 {r.source:25s} --[{r.rel_type:20s}]--> {r.target:25s} ({r.category})')

print(f'\n  Categories: {categories}')

_relationships = relationships
del mapper; gc.collect()

### 1c. Fund Mapping

In [ ]:
t0 = time.time()
from agl_security_tool.state_extraction.fund_mapper import FundMapper

fmapper = FundMapper()
fund_result = fmapper.map(source_code, TARGET)
t1 = time.time()

flows = getattr(fund_result, 'flows', []) if fund_result else []
balances = getattr(fund_result, 'balances', []) if fund_result else []

print(f'\u23f1 Time: {t1-t0:.3f}s | RAM: {mem_mb():.0f} MB')
print(f'\ud83d\udfe2 Fund Flows: {len(flows)}')
for ff in flows:
    print(f'  \ud83d\udcb8 {ff.source} \u2192 {ff.target} | token={ff.token} | amount={ff.amount}')

print(f'\n\ud83d\udfe2 Balances: {len(balances)}')
for b in balances:
    print(f'  \ud83d\udcca {b.entity}: {b.token} = {b.amount} (dir={b.direction})')

_flows = flows
_balances = balances
del fmapper, fund_result; gc.collect()

### 1d. Dynamic State Transition Model (Temporal Analysis)

Sub-modules: `execution_semantics`, `state_mutation`, `function_effects`, `temporal_graph`

In [ ]:
t0 = time.time()

# Step 1: Parse contracts (needed by temporal)
from agl_security_tool.detectors.solidity_parser import SoliditySemanticParser
parser = SoliditySemanticParser()
contracts = parser.parse(source_code, TARGET)
print(f'Parsed {len(contracts)} contracts')

# Step 2: Execution Semantics
from agl_security_tool.state_extraction.execution_semantics import ExecutionSemanticsExtractor
sem_ext = ExecutionSemanticsExtractor()
timelines = sem_ext.extract(source_code, TARGET)
print(f'ExecutionTimelines: {len(timelines)}')
for tl in timelines[:5]:
    fn = getattr(tl, 'function_name', '?')
    steps = len(getattr(tl, 'steps', []))
    print(f'  \u23f1 {fn}(): {steps} steps')

# Step 3: State Mutations
from agl_security_tool.state_extraction.state_mutation import StateMutationTracker
mut_tracker = StateMutationTracker()
mutations = mut_tracker.track(source_code, TARGET)
print(f'\nStateMutations: {len(mutations)}')
for m in mutations[:5]:
    fn = getattr(m, 'function', '?')
    var = getattr(m, 'variable', '?')
    op = getattr(m, 'operation', '?')
    print(f'  \ud83d\udd04 {fn}(): {var} [{op}]')

# Step 4: Function Effects
from agl_security_tool.state_extraction.function_effects import FunctionEffectModeler
eff_modeler = FunctionEffectModeler()
effects = eff_modeler.model(source_code, TARGET)
print(f'\nFunctionEffects: {len(effects)}')
for e in effects[:5]:
    fn = getattr(e, 'function', '?')
    reads = len(getattr(e, 'reads', []))
    writes = len(getattr(e, 'writes', []))
    calls = len(getattr(e, 'external_calls', []))
    print(f'  \u26a1 {fn}(): reads={reads} writes={writes} ext_calls={calls}')

# Step 5: Temporal Dependency Graph
from agl_security_tool.state_extraction.temporal_graph import TemporalDependencyGraph
tdg = TemporalDependencyGraph()
temporal = tdg.build(timelines, mutations, effects, contracts)

t1 = time.time()
print(f'\n\u23f1 Total Temporal Time: {t1-t0:.3f}s | RAM: {mem_mb():.0f} MB')

if temporal:
    cei_v = getattr(temporal, 'cei_violations', [])
    reent = getattr(temporal, 'reentrancy_risks', [])
    print(f'\n\ud83d\udea8 CEI Violations: {len(cei_v)}')
    for v in cei_v:
        print(f'  \u26a0\ufe0f {v}')
    print(f'\n\ud83d\udea8 Reentrancy Risks: {len(reent)}')
    for r in reent:
        print(f'  \u26a0\ufe0f {r}')

_temporal = temporal
_contracts = contracts
_timelines = timelines
_mutations = mutations
_effects = effects
del sem_ext, mut_tracker, eff_modeler, tdg; gc.collect()

### 1e. Full State Extraction Engine (orchestrator)

In [ ]:
t0 = time.time()
from agl_security_tool.state_extraction import StateExtractionEngine

state_engine = StateExtractionEngine({})
state_result = state_engine.extract(TARGET)
t1 = time.time()

g = state_result.graph
print(f'\u23f1 Full Engine Time: {t1-t0:.3f}s | RAM: {mem_mb():.0f} MB')
print(f'\ud83d\udfe2 Entities: {len(g.entities)}')
print(f'\ud83d\udfe2 Relationships: {len(g.relationships)}')
print(f'\ud83d\udfe2 Fund Flows: {len(g.fund_flows)}')
print(f'\ud83d\udfe2 Balances: {len(g.balances)}')
print(f'\ud83d\udfe2 Temporal: {g.temporal_analysis is not None}')
print(f'\ud83d\udfe2 Validation: {g.validation is not None}')
print(f'\n>>> L1\u2192L2 Action Space: {g.action_space is not None}')
print(f'>>> L2\u2192L3 Attack Simulation: {g.attack_simulation is not None}')
print(f'>>> L3\u2192L4 Search Results: {g.search_results is not None}')

# Save graph for L2/L3/L4 inspection
_graph = g

---
## 🟡 Layer 2: Action Space

**فضاء الأفعال** — استخراج + تصنيف + ربط الأفعال بالحالة

Sub-modules: `action_enumerator`, `parameter_generator`, `state_linker`, `action_classifier`, `action_graph`

In [ ]:
as_data = _graph.action_space

if as_data:
    print('\ud83d\udfe2 Action Space Generated!')
    print(f'  Type: {type(as_data).__name__}')
    
    as_graph = getattr(as_data, 'graph', None)
    actions = as_graph.actions if as_graph and hasattr(as_graph, 'actions') else []
    sequences = getattr(as_data, 'attack_sequences', [])
    surfaces = getattr(as_data, 'attack_surfaces', [])
    targets = getattr(as_data, 'high_value_targets', [])
    
    print(f'\n  \ud83c\udfaf Total Actions: {len(actions)}')
    for i, a in enumerate(actions):
        print(f'    {i+1}. {str(a)[:200]}')
    
    print(f'\n  \u2694\ufe0f Attack Sequences: {len(sequences)}')
    for s in sequences[:10]:
        print(f'    \u2022 {str(s)[:200]}')
    
    print(f'\n  \ud83d\udee1\ufe0f Attack Surfaces: {len(surfaces)}')
    for s in surfaces[:10]:
        print(f'    \u2022 {str(s)[:200]}')
    
    print(f'\n  \ud83d\udca3 High-Value Targets: {len(targets)}')
    for t in targets[:10]:
        print(f'    \u2022 {t}')
    
    if as_graph:
        nodes = getattr(as_graph, 'nodes', [])
        edges = getattr(as_graph, 'edges', [])
        paths = getattr(as_graph, 'attack_paths', [])
        print(f'\n  \ud83d\udcc8 Action Graph:')
        print(f'    Nodes: {len(nodes)}')
        print(f'    Edges: {len(edges)}')
        print(f'    Attack Paths: {len(paths)}')
        for p in paths[:5]:
            print(f'      \u27a1\ufe0f {str(p)[:200]}')
else:
    print('\u274c Action Space NOT generated from L1')
    print('Testing Action Space builder directly...')
    
    from agl_security_tool.action_space.builder import ActionSpaceBuilder
    builder = ActionSpaceBuilder()
    as_result = builder.build(_graph)
    print(f'Direct build result: {type(as_result).__name__}')
    print(f'  {as_result}')

---
## 🟠 Layer 3: Attack Engine

**محاكاة الهجمات** — تنفيذ دلالي + حساب الربح

Sub-modules: `action_executor`, `profit_calculator`, `state_mutator`, `economic_engine`

In [ ]:
sim = _graph.attack_simulation

if sim:
    print('\ud83d\udfe2 Attack Simulation Completed!')
    print(f'  Type: {type(sim).__name__}')
    print(f'  \u2694\ufe0f Total Sequences Tested: {getattr(sim, "total_sequences_tested", 0)}')
    print(f'  \ud83d\udcb0 Profitable Attacks: {getattr(sim, "profitable_attacks", 0)}')
    print(f'  \ud83d\udcb5 Total Profit (USD): ${getattr(sim, "total_profit_usd", 0):.2f}')
    print(f'  \ud83c\udfaf Attack Types: {getattr(sim, "attack_types_found", {})}')
    print(f'  \ud83d\udcca Severity Distribution: {getattr(sim, "severity_distribution", {})}')
    print(f'  \u23f1 Execution Time: {getattr(sim, "execution_time_ms", 0):.0f}ms')
    
    best = getattr(sim, 'best_attack', None)
    if best:
        print(f'\n  \ud83c\udfc6 Best Attack:')
        print(f'    Type: {getattr(best, "attack_type", "?")}')
        print(f'    Profitable: {getattr(best, "is_profitable", False)}')
        print(f'    Net Profit: ${getattr(best, "net_profit_usd", 0):.2f}')
        steps = getattr(best, 'steps', [])
        print(f'    Steps: {len(steps)}')
        for i, s in enumerate(steps):
            print(f'      {i+1}. {str(s)[:200]}')
    
    all_results = getattr(sim, 'all_results', [])
    if all_results:
        print(f'\n  \ud83d\udccb All Attack Results ({len(all_results)}):')
        for ar in all_results[:15]:
            profit = getattr(ar, 'net_profit_usd', 0)
            atype = getattr(ar, 'attack_type', '?')
            is_p = getattr(ar, 'is_profitable', False)
            icon = '\ud83d\udcb0' if is_p else '\u274c'
            print(f'    {icon} {atype}: ${profit:.2f} profitable={is_p}')
else:
    print('\u274c Attack Simulation NOT generated')
    print('This means L2 did not produce enough actions for L3 to simulate.')

---
## 🔵 Layer 4: Search Engine

**بحث ذكي** — MCTS + Beam Search + Evolutionary

Sub-modules: `guided_search`, `weakness_detector`, `profit_gradient`, `heuristic_prioritizer`

In [ ]:
sr = _graph.search_results

if sr:
    print('\ud83d\udfe2 Search Engine Completed!')
    print(f'  Type: {type(sr).__name__}')
    profitable = getattr(sr, 'profitable_sequences', [])
    total_eval = getattr(sr, 'total_evaluated', 0)
    strategies = getattr(sr, 'strategies_used', [])
    
    print(f'  \ud83d\udd0d Total Evaluated: {total_eval}')
    print(f'  \ud83d\udcb0 Profitable Sequences: {len(profitable)}')
    print(f'  \ud83e\udde0 Strategies Used: {strategies}')
    
    for i, seq in enumerate(profitable[:10]):
        print(f'\n    Sequence {i+1}: {str(seq)[:300]}')
else:
    print('\u274c Search Results NOT generated')
    print('This means L3 did not find enough profitable patterns for L4 to optimize.')

---
## 🟣 Layer 5: Detectors (Independent)

**22 كاشف مستقل** — Reentrancy, Access Control, DeFi, Common, Token

Sub-modules: `solidity_parser`, `reentrancy`, `access_control`, `defi`, `common`, `token`

In [ ]:
t0 = time.time()
from agl_security_tool.detectors import DetectorRunner
from agl_security_tool.detectors.solidity_parser import SoliditySemanticParser

parser = SoliditySemanticParser()
runner = DetectorRunner()

contracts = parser.parse(source_code, TARGET)
print(f'\ud83d\udfe2 Parsed {len(contracts)} contracts\n')

for c in contracts:
    funcs = getattr(c, 'functions', [])
    svars = getattr(c, 'state_variables', [])
    ext_calls = getattr(c, 'external_calls', [])
    print(f'  Contract: {c.name}')
    print(f'    Functions ({len(funcs)}):')
    for fn in funcs:
        if isinstance(fn, str):
            print(f'      \u2022 {fn}()')
            continue
        fname = getattr(fn, 'name', str(fn))
        mods = getattr(fn, 'modifiers', [])
        reads = len(getattr(fn, 'state_reads', []))
        writes = len(getattr(fn, 'state_writes', []))
        calls = len(getattr(fn, 'external_calls', []))
        print(f'      \u2022 {fname}() mods={mods} reads={reads} writes={writes} ext_calls={calls}')
    print(f'    State Variables ({len(svars)}):')
    for sv in svars:
        print(f'      \u2022 {sv.name}: {sv.var_type}')
    print(f'    External Calls: {len(ext_calls)}')
    print()

det_findings = runner.run(contracts)
t1 = time.time()

print(f'\u23f1 Time: {t1-t0:.3f}s | RAM: {mem_mb():.0f} MB')
print(f'\n\ud83d\udea8 Detector Findings: {len(det_findings)}\n')

severity_counts = {}
for df in det_findings:
    d = df.to_dict()
    sev = d.get('severity', '?').upper()
    severity_counts[sev] = severity_counts.get(sev, 0) + 1
    title = d.get('title', '?')
    line = d.get('line', '?')
    det_id = d.get('detector_id', '?')
    print(f'  [{sev:>8}] L{line}: {title}')
    desc = d.get('description', '')[:150]
    if desc:
        print(f'             {desc}')
    print()

print(f'\ud83d\udcca Summary: {severity_counts}')

---
## ?\udcca Pipeline Summary

In [ ]:
print('=' * 60)
print('  PIPELINE INTERCONNECTION SUMMARY')
print('=' * 60)
print(f'  L1 (State Extraction):')
print(f'    Entities:      {len(_graph.entities)}')
print(f'    Relationships: {len(_graph.relationships)}')
print(f'    Fund Flows:    {len(_graph.fund_flows)}')
print(f'    Balances:      {len(_graph.balances)}')
print(f'    Temporal:      {"YES" if _graph.temporal_analysis else "NO"}')
print(f'    Validation:    {"YES" if _graph.validation else "NO"}')
print()
print(f'  L1 \u2192 L2 (Action Space):  {"\u2705 CONNECTED" if _graph.action_space else "\u274c NOT CONNECTED"}')
print(f'  L2 \u2192 L3 (Attack Engine): {"\u2705 CONNECTED" if _graph.attack_simulation else "\u274c NOT CONNECTED"}')
print(f'  L3 \u2192 L4 (Search Engine): {"\u2705 CONNECTED" if _graph.search_results else "\u274c NOT CONNECTED"}')
print(f'  L5 (Detectors):            \u2705 INDEPENDENT (ran separately)')
print()
print(f'  \ud83d\udcca RAM Usage: {mem_mb():.0f} MB')
print(f'  \ud83d\udcca Free RAM: {psutil.virtual_memory().available/1024**3:.1f} GB')
print('=' * 60)

---
## ?\udfaf Bonus: Test All 4 Contracts

In [ ]:
all_targets = [
    'test_project/src/Vault.sol',
    'test_project/src/VaultToken.sol',
    'test_project/src/VaultFactory.sol',
    'test_project/src/RewardDistributor.sol',
]

from agl_security_tool.state_extraction import StateExtractionEngine
from agl_security_tool.detectors import DetectorRunner
from agl_security_tool.detectors.solidity_parser import SoliditySemanticParser

for target in all_targets:
    if not os.path.exists(target):
        print(f'\u274c {target} not found')
        continue
    
    print(f'\n{"=" * 60}')
    print(f'  \ud83d\udcc4 {target}')
    print(f'{"=" * 60}')
    
    # L1-L4 Pipeline
    t0 = time.time()
    engine = StateExtractionEngine({})
    result = engine.extract(target)
    g = result.graph
    t1 = time.time()
    
    print(f'  L1: entities={len(g.entities)}, rels={len(g.relationships)}, flows={len(g.fund_flows)} ({t1-t0:.2f}s)')
    print(f'  L2: {"YES" if g.action_space else "NO"}')
    print(f'  L3: {"YES" if g.attack_simulation else "NO"}')
    print(f'  L4: {"YES" if g.search_results else "NO"}')
    
    # L5 Detectors
    with open(target, 'r') as f:
        code = f.read()
    p = SoliditySemanticParser()
    r = DetectorRunner()
    contracts = p.parse(code, target)
    findings = r.run(contracts)
    
    print(f'  L5: {len(findings)} findings')
    for df in findings:
        d = df.to_dict()
        print(f'    [{d["severity"].upper():>8}] L{d["line"]}: {d["title"]}')
    
    del engine, result, g
    gc.collect()

print(f'\n\ud83c\udfc1 All done! RAM: {mem_mb():.0f} MB')